# QLoRA Fine-Tuning (Colab)

Welcome! In this notebook you will **fine-tune a small language model on a free GPU** and then
measure whether the fine-tune actually helped. No prior fine-tuning experience needed.

**What's about to happen, in plain words:**
1. **Install** — grab this repo's code plus the GPU training libraries.
2. **Data** — generate a few hundred tiny practice tasks (add numbers, reverse strings) and set
   10% aside as an "exam" the model never studies from.
3. **Train** — teach the model those tasks with **QLoRA**. Instead of changing the whole model
   (huge, expensive), QLoRA freezes it, loads it in compressed 4-bit form, and trains only tiny
   **adapter** layers bolted on top — a few MB of new weights. That's why a free Colab T4 GPU
   is enough.
4. **Evaluate** — quiz both the original ("base") model and the fine-tuned one on the held-out
   exam. If the tuned model doesn't score higher, the fine-tune wasn't worth it.

**Before running anything:** in the Colab menu choose **Runtime → Change runtime type → T4 GPU**.
Then run the cells top to bottom (Shift+Enter). Total time: roughly 10–15 minutes.

> Lost at any point? `docs/code-walkthrough.md` in the repo explains every file and function
> this notebook uses, in plain English.

## Step 1 — Install

This installs the `lora-finetune-lab` package straight from GitHub, with its `[train]` extra —
the heavy GPU libraries (`torch`, `transformers`, `peft`, `trl`, `bitsandbytes`) that the repo
deliberately keeps out of its normal install. The `print` at the end confirms Colab really gave
us a GPU: it must say `CUDA available: True`. If it says `False`, fix the runtime type (see
above) before continuing — training on CPU would take hours.

In [ ]:
# 1. Install the repo (with the GPU training stack) + dependencies.
!pip -q install "git+https://github.com/ArunRyzen/lora-finetune-lab.git#egg=lora-finetune-lab[train]"
import torch; print('CUDA available:', torch.cuda.is_available())

## Step 2 — Build the dataset

Fine-tuning data is just a list of examples: an *instruction*, an optional *input*, and the
*output* we want. Here we generate 400 synthetic ones (simple arithmetic / uppercase / reverse
tasks with known answers) and split off 10% as a **validation set** — questions the model will
never see during training, so the final comparison is a fair exam and not a memory test.

`format_text` then turns each example into one string in a fixed chat layout
(`<|system|>… <|user|>… <|assistant|>…`). The printed sample shows the *exact* text the model
will learn from — always worth eyeballing before you train.

In [ ]:
# 2. Build data: synthetic instruction examples (swap in your own JSONL for real tasks).
from lora_finetune_lab.dataset import generate_synthetic, split
from lora_finetune_lab.prompts import format_text

examples = generate_synthetic(400)
train, val = split(examples, val_fraction=0.1)
train_texts = [format_text(ex) for ex in train]
print(len(train), 'train /', len(val), 'val')
print(train_texts[0])

## Step 3 — Train with QLoRA

One `TrainingConfig` object holds every knob (model name, adapter size, learning rate, epochs —
see `src/lora_finetune_lab/config.py` for a plain-English tour of each). `run_training` then:

1. downloads the base model **compressed to 4-bit** (the "Q" in QLoRA) so it fits in GPU memory,
2. bolts small trainable **LoRA adapter** matrices onto its attention layers,
3. trains *only those adapters* on our 360 examples, and saves them to `outputs/adapter`.

The saved adapter is a few **megabytes** — not a copy of the whole model. Expect a few minutes
of progress logs; the `loss` number trending downward means the model is learning.

In [ ]:
# 3. Configure and run QLoRA training (loads the base in 4-bit, trains LoRA adapters).
from lora_finetune_lab.config import TrainingConfig
from lora_finetune_lab.train import run_training

cfg = TrainingConfig(base_model='Qwen/Qwen2.5-0.5B-Instruct', epochs=2, output_dir='outputs/adapter')
adapter_dir = run_training(cfg, train_texts)
print('Adapter saved to', adapter_dir)

## Step 4 — The verdict: base vs fine-tuned

Now the exam. We load the model **twice**: once as the untouched base, and once with our trained
adapter attached on top (`PeftModel.from_pretrained` = "base model + our add-on").

The small `HFModel` wrapper just gives each model a `generate(prompt) -> str` method — that's
the repo's `Model` protocol, the same interface the offline demo in `lora eval` uses. Thanks to
that, the *same* `accuracy` function from `evaluation.py` grades both models on the held-out
validation set: each model gets the question without the answer, and scores a point only if the
expected answer appears in its reply.

In [ ]:
# 4. Evaluate base vs fine-tuned on the held-out val set.
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from lora_finetune_lab.evaluation import accuracy

tok = AutoTokenizer.from_pretrained(cfg.base_model)
base = AutoModelForCausalLM.from_pretrained(cfg.base_model, device_map='auto')

class HFModel:
    def __init__(self, model):
        self.model = model
    def generate(self, prompt):
        ids = tok(prompt, return_tensors='pt').to(self.model.device)
        out = self.model.generate(**ids, max_new_tokens=16, do_sample=False)
        return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

# IMPORTANT: grade the base BEFORE attaching the adapter. PeftModel.from_pretrained
# modifies the wrapped model in place, so grading "base" afterwards would secretly
# include the fine-tune and the comparison would lie.
print('base   :', accuracy(HFModel(base), val))

tuned = PeftModel.from_pretrained(base, adapter_dir)
print('tuned  :', accuracy(HFModel(tuned), val))


## Read the result, then decide

You should see the tuned model score well above the base. But the score itself isn't the lesson —
the *decision habit* is:

- **Tuned beats base on the held-out set** → the fine-tune earned its keep.
- **Tuned ≈ base**, or a cheaper prompt/RAG change would have matched it → don't ship it.

And remember what fine-tuning can and cannot fix: **fine-tuning changes behaviour** (skills,
format, style — like these tasks), while **RAG adds knowledge** (facts the model doesn't have).
If your real problem is missing knowledge, no amount of QLoRA will solve it. The full decision
framework is in [`docs/when-to-finetune.md`](../docs/when-to-finetune.md), and
[`docs/code-walkthrough.md`](../docs/code-walkthrough.md) walks through where this comparison
lives in the code.